In [11]:
from bioservices import KEGG
import os

# 저장 경로
output_dir = r"C:\Users\meg57\OneDrive\바탕 화면\은기\진행중\PFAS_transcriptomics_read-across\classification\Raw-files for GSEA"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "zebrafish_KEGG_EntrezID_only.txt")

# KEGG 초기화
kegg = KEGG()

# ✅ 정확한 인자 조합: pathway → dre genes
link_data = kegg.link("dre", "pathway").strip().split("\n")

# pathway별로 유전자 리스트 정리
mapping = {}
for line in link_data:
    path, gene = line.split("\t")
    path_id = path.replace("path:", "")     # dre04110
    gene_id = gene.replace("dre:", "")      # Entrez Gene ID
    mapping.setdefault(path_id, []).append(gene_id)

# 저장
with open(output_path, "w", encoding="utf-8") as f:
    for pid, genes in mapping.items():
        f.write(f"{pid}\t" + "\t".join(genes) + "\n")

print(f"✅ {len(mapping)}개의 KEGG pathway가 저장되었습니다 → {output_path}")


✅ 189개의 KEGG pathway가 저장되었습니다 → C:\Users\meg57\OneDrive\바탕 화면\은기\진행중\PFAS_transcriptomics_read-across\classification\Raw-files for GSEA\zebrafish_KEGG_EntrezID_only.txt


In [15]:
from mygene import MyGeneInfo
import pandas as pd

# 1. Entrez ID 리스트 (예시: all_entrez = ['30590', '30591', ...])
mg = MyGeneInfo()
query_result = mg.querymany(list(all_entrez), scopes="entrezgene", fields="ensembl.gene", species=7955, as_dataframe=True)

# ✅ 인덱스를 열로 복사
query_result["query"] = query_result.index

# 2. 마스킹
if 'notfound' in query_result.columns:
    notfound_col = query_result['notfound'].fillna(False).astype(bool)
    mask = (~notfound_col) & (query_result['ensembl.gene'].notna())
else:
    mask = query_result['ensembl.gene'].notna()

query_result_clean = query_result[mask].copy()

# 3. Ensembl ID 추출
def extract_ensembl(x):
    if isinstance(x, list):
        return [i['gene'] for i in x if 'gene' in i]
    elif isinstance(x, dict) and 'gene' in x:
        return [x['gene']]
    else:
        return []

query_result_clean['EnsemblIDs'] = query_result_clean['ensembl.gene'].apply(extract_ensembl)

# 4. Entrez ↔ Ensembl 매핑 딕셔너리 생성
mapping_df = query_result_clean[['query', 'EnsemblIDs']].explode('EnsemblIDs').dropna().drop_duplicates()
entrez_to_ensembl = dict(zip(mapping_df['query'].astype(str), mapping_df['EnsemblIDs']))

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
39 input query terms found no hit:	['100537262', '137487191', '560917', '137496528', '100005220', '137487188', '110440104', '110438480'
C:\Users\meg57\AppData\Local\Temp\ipykernel_22784\275165721.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  notfound_col = query_result['notfound'].fillna(False).astype(bool)


In [22]:
query_result_df = query_result.copy()

# 유효한 Ensembl gene만 필터링
notfound_col = query_result_df['notfound'].fillna(False).astype(bool)

query_result_clean = query_result_df[
    query_result_df['ensembl.gene'].notna() & (~notfound_col)
]

# Ensembl ID 추출 함수 정의
def extract_ensembl(x):
    if isinstance(x, dict) and 'gene' in x:
        return [x['gene']]
    elif isinstance(x, list):
        return [i['gene'] for i in x if 'gene' in i]
    else:
        return []

query_result_clean['EnsemblIDs'] = query_result_clean['ensembl.gene'].apply(extract_ensembl)

# Entrez ↔ Ensembl 매핑 테이블 생성
mapping_df = query_result_clean[['query', 'EnsemblIDs']].explode('EnsemblIDs').dropna().drop_duplicates()
entrez_to_ensembl = dict(zip(mapping_df['query'].astype(str), mapping_df['EnsemblIDs']))

C:\Users\meg57\AppData\Local\Temp\ipykernel_22784\3049200949.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  notfound_col = query_result_df['notfound'].fillna(False).astype(bool)
C:\Users\meg57\AppData\Local\Temp\ipykernel_22784\3049200949.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  query_result_clean['EnsemblIDs'] = query_result_clean['ensembl.gene'].apply(extract_ensembl)


In [31]:
import pandas as pd
from bioservices import BioMart
from io import StringIO
import os

# 🔧 파일 경로
input_path = r"C:\Users\meg57\OneDrive\바탕 화면\은기\진행중\PFAS_transcriptomics_read-across\classification\Raw-files for GSEA\zebrafish_KEGG_EntrezID_only.txt"
output_path = input_path.replace("EntrezID_only", "EnsemblID_only.txt")

# 🔄 Entrez ID → Ensembl ID 매핑을 위한 함수
def get_entrez_to_ensembl(entrez_ids):
    bm = BioMart(host="www.ensembl.org")
    bm.dataset = "drerio_gene_ensembl"
    
    query_xml = f"""
    <?xml version="1.0" encoding="UTF-8"?>
    <!DOCTYPE Query>
    <Query virtualSchemaName = "default" formatter = "TSV" header = "0" uniqueRows = "1" count = "" datasetConfigVersion = "0.6">
        <Dataset name = "drerio_gene_ensembl" interface = "default">
            <Filter name = "entrezgene_id" value = "{','.join(entrez_ids)}"/>
            <Attribute name = "ensembl_gene_id"/>
            <Attribute name = "entrezgene_id"/>
        </Dataset>
    </Query>
    """
    result = bm.query(query_xml)
    df = pd.read_csv(StringIO(result), sep="\t", header=None)
    df.columns = ["Ensembl_ID", "Entrez_ID"]
    return dict(zip(df["Entrez_ID"].astype(str), df["Ensembl_ID"]))

# 📖 KEGG Entrez 파일 로드
with open(input_path, "r") as infile:
    lines = infile.readlines()

# 📄 출력 파일 준비
with open(output_path, "w") as outfile:
    for line in lines:
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        pathway_id = parts[0]
        entrez_ids = parts[1:]
        
        # Ensembl 매핑 (작게 나눠야 timeout 피함)
        mapping = get_entrez_to_ensembl(entrez_ids)
        ensembl_ids = [mapping[eid] for eid in entrez_ids if eid in mapping]
        
        # 저장
        if ensembl_ids:
            line_out = pathway_id + "\t" + "\t".join(ensembl_ids) + "\n"
            outfile.write(line_out)

print(f"✅ 변환 완료! 저장 위치: {output_path}")

✅ 변환 완료! 저장 위치: C:\Users\meg57\OneDrive\바탕 화면\은기\진행중\PFAS_transcriptomics_read-across\classification\Raw-files for GSEA\zebrafish_KEGG_EnsemblID_only.txt.txt


In [44]:
import pandas as pd
import os
from collections import defaultdict

# 🔧 경로 설정
base_path = r"C:\Users\meg57\OneDrive\바탕 화면\은기\진행중\PFAS_transcriptomics_read-across\classification\Raw-files for GSEA"
go_tsv_file = os.path.join(base_path, "GO for danio rerio.txt")  # 주신 GO 포맷
kegg_file = os.path.join(base_path, "zebrafish_KEGG_EnsemblID_only.txt")
output_gmt = os.path.join(base_path, "zebrafish_GO_KEGG_EnsemblID_merged.gmt")

# ✅ GO: GO term ID 기준으로 유전자 ID 모으기
go_df = pd.read_csv(go_tsv_file, sep="\t")
go_groups = go_df.groupby("GO term accession")["Gene stable ID"].apply(set).to_dict()

# ✅ KEGG: line parsing
kegg_dict = {}
with open(kegg_file, "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            pathway_id = parts[0]
            genes = parts[1:]
            kegg_dict[pathway_id] = set(genes)

# ✅ .gmt 라인 생성
gmt_lines = []

# GO terms
for go_id, gene_set in go_groups.items():
    gmt_line = [go_id, "GeneOntology"] + sorted(gene_set)
    gmt_lines.append("\t".join(gmt_line))

# KEGG pathways
for kegg_id, gene_set in kegg_dict.items():
    gmt_line = [kegg_id, "KEGGpathway"] + sorted(gene_set)
    gmt_lines.append("\t".join(gmt_line))

# ✅ 저장
with open(output_gmt, "w") as f:
    for line in gmt_lines:
        f.write(line + "\n")

print(f"✅ GMT 파일 생성 완료: {output_gmt}")

C:\Users\meg57\AppData\Local\Temp\ipykernel_22784\1379494188.py:12: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  go_df = pd.read_csv(go_tsv_file, sep="\t")


✅ GMT 파일 생성 완료: C:\Users\meg57\OneDrive\바탕 화면\은기\진행중\PFAS_transcriptomics_read-across\classification\Raw-files for GSEA\zebrafish_GO_KEGG_EnsemblID_merged.gmt
